# Scraping Ulasan Google Play - Bank Jago

Notebook ini melakukan scraping ulasan aplikasi Bank Jago dari Google Play Store.
Output: `data/raw/reviews.csv` (~10.000 baris).

In [1]:
import csv
import os
import time
from google_play_scraper import reviews_all, Sort

In [2]:
APP_ID = "com.jago.digitalBanking"
OUTPUT_FILE = "../data/raw/reviews.csv"
TARGET_COUNT = 10000
LANG = "id"
COUNTRY = "id"

In [3]:
os.makedirs('../data/raw', exist_ok=True)

In [4]:
print(f"Scraping {TARGET_COUNT} reviews for {APP_ID}...")
result = reviews_all(APP_ID, sleep_milliseconds=1000, lang=LANG, country=COUNTRY, sort=Sort.NEWEST)
print(f"Fetched {len(result)} reviews total")

Scraping 10000 reviews for com.jago.digitalBanking...


Fetched 70227 reviews total


In [5]:
# Deduplikasi berdasarkan reviewId
id_filter = set()
unique = []
for r in result:
    if r['reviewId'] not in id_filter:
        id_filter.add(r['reviewId'])
        unique.append(r)
print(f"Unique reviews: {len(unique)}")

Unique reviews: 70227


In [6]:
# Simpan ke CSV (sampel pertama TARGET_COUNT)
sample = unique[:TARGET_COUNT]
with open(OUTPUT_FILE, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['reviewId', 'content', 'score', 'at', 'userName'])
    writer.writeheader()
    for r in sample:
        writer.writerow({
            'reviewId': r['reviewId'],
            'content': r['content'],
            'score': r['score'],
            'at': r['at'].isoformat() if hasattr(r['at'], 'isoformat') else r['at'],
            'userName': r['userName'],
        })
print(f'Saved {len(sample)} reviews to {OUTPUT_FILE}')

Saved 10000 reviews to ../data/raw/reviews.csv


In [7]:
import pandas as pd
df = pd.read_csv(OUTPUT_FILE)
print(f'Dataset: {df.shape[0]} rows, {df.shape[1]} columns')
print(f'\nScore distribution:')
print(df['score'].value_counts().sort_index())
print(f"\nPositive: {(df['score']>=4).sum()}, Neutral: {(df['score']==3).sum()}, Negative: {(df['score']<=2).sum()}")
df.head()

Dataset: 10000 rows, 5 columns

Score distribution:
score
1    2360
2     291
3     315
4     421
5    6613
Name: count, dtype: int64

Positive: 7034, Neutral: 315, Negative: 2651


,reviewId,content,score,at,userName
0,a421aba9-949f-42e9-bf42-f3fbb21d12a6,sangat membantu kami . kami sangat puas,5,2026-05-28T01:34:29,Pengguna Google
1,0c651415-a589-4b43-bb58-9e6524a4097c,sering error salah password dan pin padahal su...,4,2026-05-28T01:13:43,Pengguna Google
2,5e2c25cf-774b-4298-a385-86924c4191cf,saya ingin mengganti nomor HP akun karena kart...,5,2026-05-27T23:32:56,Pengguna Google
3,b854651f-ef43-4167-9828-769980b1614e,sangat membantu,5,2026-05-27T23:19:19,Pengguna Google
4,f1f0bc9c-7003-44ac-b65d-d18b3f514b6d,"pembayaran, transfer pembelanjaan Qris sangat ...",5,2026-05-27T21:41:54,Pengguna Google


## Verifikasi Output

In [8]:
# Verifikasi file exists dan valid
assert os.path.exists(OUTPUT_FILE), f'File {OUTPUT_FILE} tidak ditemukan!'
assert len(df) == len(sample), 'Jumlah baris CSV tidak sesuai dengan sample!'
print(f'OK: {OUTPUT_FILE} valid, {len(df)} rows')

OK: ../data/raw/reviews.csv valid, 10000 rows
